# **ClinTrialPredict Estimation: Operational Benchmark Layer**

This notebook is the central reproducible analytical notebook for the deterministic operational benchmark layer used by Simulation Mode. It now brings together the logic that was previously split across:

```text
notebooks/estimation.ipynb
notebooks/estimation_sites.ipynb
```

The notebook explains how the benchmark pieces fit together:

1. Planned Enrollment benchmark from completed trials with reliable positive `ACTUAL` enrollment.
2. Planned Site Count benchmark from completed trials with positive `number_of_facilities`.
3. Combined operational benchmark with enrollment, site-count proxy, and patients-per-site percentiles in one cohort-aligned artifact.
4. Revised non-completed trial defaulting rule for `planned_sites`, where current registry facility count is a lower-bound/context value, not automatically the planned-sites estimate.

The benchmarks are deterministic reference signals. They do **not** enter XGBoost, SHAP, therapeutic-area calibration, audit parity, `/predict`, prediction payloads, or API contracts. They do not modify the Completion Score.

## **Rules Implemented So Far**

### Planned Enrollment

- Target population: completed trials with positive `ACTUAL` enrollment.
- Historical artifact: `frontend/data/enrollment_benchmarks_v1.csv`.
- Active runtime artifact: `frontend/data/operational_benchmarks_v1.csv`.
- Active runtime utility: `src/operational_benchmarks.py`.
- Opening default rules:
  - If planned/estimated enrollment is available, use it as `planned_value`.
  - If the trial is completed and actual enrollment is available, use it as `final_observed_value`.
  - If the trial is non-completed and planned/estimated enrollment is missing, treat actual/current enrollment as `observed_lower_bound` when available.
  - If benchmark P50 is available, use it as `model_default`.
  - For non-completed trials with both observed lower-bound and benchmark P50, use `max(observed_lower_bound, model_default)`.
  - If the user edits the value, source becomes `user_scenario`.
- Classification bands:
  - `< P25`: `below_benchmark`
  - `P25` through `P75`: `typical`
  - above `P75` through `P90`: `ambitious`
  - above `P90`: `above_benchmark_high`

### Planned Site Count

- Source field: `number_of_facilities`.
- Source caveat: this is a registry-derived aggregate facility-count proxy.
- It is not true planned sites, true actual activated sites, or true estimated sites.
- Target population: completed trials with positive `number_of_facilities`.
- Historical artifact: `frontend/data/site_benchmarks_v1.csv`.
- Active runtime artifact: `frontend/data/operational_benchmarks_v1.csv`.
- Active runtime utility: `src/operational_benchmarks.py`.

### Combined Operational Benchmark

- Artifact: `frontend/data/operational_benchmarks_v1.csv`.
- Runtime utility: `src/operational_benchmarks.py`.
- Cohort hierarchy for all metrics:
  1. `phase + indication + rare disease flag`
  2. `phase + therapeutic area + rare disease flag`
  3. `phase + therapeutic area`
  4. `phase only`
- Minimum confident cohort threshold: `n >= 50`.
- Metrics carried in one artifact:
  - enrollment percentiles and `enrollment_n`
  - site-count proxy percentiles and `site_count_n`
  - patients-per-site percentiles and `patients_per_site_n`
- Patients-per-site target: completed trials with positive `ACTUAL` enrollment and positive `number_of_facilities`.
- Patients-per-site formula: `enrollment / number_of_facilities`.

### Revised `planned_sites` Default Rule

For completed trials:

```text
planned_sites = completed_registry_facility_count
```

For non-completed trials:

```text
planned_sites = max(
    current_registry_facility_count_proxy,
    planned_enrollment / patients_per_site_p50
)
```



Use pure `site_count_benchmark_p50` only as fallback when patients-per-site P50 is unavailable.

For non-completed trials, `number_of_facilities` is displayed/stored as `current_registry_facility_count_proxy` context and lower-bound evidence. It is not treated as true planned or final site count.

## **Reproducibility Workflow**

Run these from the repository root.

1. Rebuild the active combined operational artifact:

   ```bash
   python scripts/build_operational_benchmarks.py
   ```

3. Validate artifacts and runtime helpers:

   ```bash
   python scripts/check_operational_benchmarks.py
   ```

4. Optional compile check after code changes:

   ```bash
   python -m py_compile \
     scripts/build_enrollment_benchmarks.py \
     scripts/build_site_benchmarks.py \
     scripts/build_operational_benchmarks.py \
     scripts/check_enrollment_benchmarks.py \
     scripts/check_site_benchmarks.py \
     scripts/check_operational_benchmarks.py \
     src/enrollment_benchmarks.py \
     src/site_benchmarks.py \
     src/operational_benchmarks.py
   ```

Then run this notebook to inspect and understand the regenerated outputs end to end.

The notebook reproduces and inspects calculations in sections such as:

```text
BENCHMARK_COHORTS
SITES_BENCHMARK_COHORTS
OPERATIONAL_BENCHMARK_ARTIFACT
VALIDATION
```


#### <REF:ENV_CONFIG>
> #### **1. Development Environment Configuration**
>
> Configure notebook behavior, reproducibility defaults, warning filters, display settings, and lightweight audit helpers. This follows the clear sectioned style of `notebooks/production_01.ipynb` while keeping this notebook analytical and read-mostly.


In [1]:
# <REF:ENV_CONFIG_CODE>
%load_ext autoreload
%autoreload 2

import json
import math
import random
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

RANDOM_STATE = 42
NOTEBOOK_VERSION = "operational_benchmark_layer_v1"
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Random state: {RANDOM_STATE}")
# <REF:/ENV_CONFIG_CODE>


Notebook version: operational_benchmark_layer_v1
Random state: 42


#### <REF:PATH_RESOLUTION>
> #### **2. Project Path Resolution and Artifact Locations**
>
> Resolve the repository root dynamically, add it to `PYTHONPATH`, and define the source, artifact, script, and runtime utility paths used by Phase 1.
>
> Runtime convention: the compact benchmark CSV lives under `frontend/data/`, a path already copied into the app image. Runtime lookup must not load `data/data_clinpred.csv`.


In [2]:
# <REF:PATH_RESOLUTION_CODE>
import sys

current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise FileNotFoundError("Could not find project root containing 'src'")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_CLINPRED_PATH = project_root / "data" / "data_clinpred.csv"

ARTIFACT_PATH = project_root / "frontend" / "data" / "enrollment_benchmarks_v1.csv"
REPORT_PATH = project_root / "frontend" / "data" / "enrollment_benchmarks_v1_report.json"
BUILD_SCRIPT_PATH = project_root / "scripts" / "build_enrollment_benchmarks.py"
CHECK_SCRIPT_PATH = project_root / "scripts" / "check_enrollment_benchmarks.py"
RUNTIME_UTILITY_PATH = project_root / "src" / "enrollment_benchmarks.py"

SITE_ARTIFACT_PATH = project_root / "frontend" / "data" / "site_benchmarks_v1.csv"
SITE_REPORT_PATH = project_root / "frontend" / "data" / "site_benchmarks_v1_report.json"
SITE_BUILD_SCRIPT_PATH = project_root / "scripts" / "build_site_benchmarks.py"
SITE_CHECK_SCRIPT_PATH = project_root / "scripts" / "check_site_benchmarks.py"
SITE_RUNTIME_UTILITY_PATH = project_root / "src" / "site_benchmarks.py"

OPERATIONAL_ARTIFACT_PATH = project_root / "frontend" / "data" / "operational_benchmarks_v1.csv"
OPERATIONAL_REPORT_PATH = project_root / "frontend" / "data" / "operational_benchmarks_v1_report.json"
OPERATIONAL_BUILD_SCRIPT_PATH = project_root / "scripts" / "build_operational_benchmarks.py"
OPERATIONAL_CHECK_SCRIPT_PATH = project_root / "scripts" / "check_operational_benchmarks.py"
OPERATIONAL_RUNTIME_UTILITY_PATH = project_root / "src" / "operational_benchmarks.py"

PATHS = {
    "source_data": DATA_CLINPRED_PATH,
    "enrollment_artifact": ARTIFACT_PATH,
    "enrollment_report": REPORT_PATH,
    "enrollment_build_script": BUILD_SCRIPT_PATH,
    "enrollment_check_script": CHECK_SCRIPT_PATH,
    "enrollment_runtime_utility": RUNTIME_UTILITY_PATH,
    "site_artifact": SITE_ARTIFACT_PATH,
    "site_report": SITE_REPORT_PATH,
    "site_build_script": SITE_BUILD_SCRIPT_PATH,
    "site_check_script": SITE_CHECK_SCRIPT_PATH,
    "site_runtime_utility": SITE_RUNTIME_UTILITY_PATH,
    "operational_artifact": OPERATIONAL_ARTIFACT_PATH,
    "operational_report": OPERATIONAL_REPORT_PATH,
    "operational_build_script": OPERATIONAL_BUILD_SCRIPT_PATH,
    "operational_check_script": OPERATIONAL_CHECK_SCRIPT_PATH,
    "operational_runtime_utility": OPERATIONAL_RUNTIME_UTILITY_PATH,
}

print(f"Project Root: {project_root}")
for label, path in PATHS.items():
    print(f"{label:<30} exists={path.exists()}  {path.relative_to(project_root)}")
# <REF:/PATH_RESOLUTION_CODE>


Project Root: /home/delaunan/code/delaunan/clintrialpredict
source_data                    exists=True  data/data_clinpred.csv
enrollment_artifact            exists=True  frontend/data/enrollment_benchmarks_v1.csv
enrollment_report              exists=True  frontend/data/enrollment_benchmarks_v1_report.json
enrollment_build_script        exists=True  scripts/build_enrollment_benchmarks.py
enrollment_check_script        exists=True  scripts/check_enrollment_benchmarks.py
enrollment_runtime_utility     exists=True  src/enrollment_benchmarks.py
site_artifact                  exists=True  frontend/data/site_benchmarks_v1.csv
site_report                    exists=True  frontend/data/site_benchmarks_v1_report.json
site_build_script              exists=True  scripts/build_site_benchmarks.py
site_check_script              exists=True  scripts/check_site_benchmarks.py
site_runtime_utility           exists=True  src/site_benchmarks.py
operational_artifact           exists=True  frontend/data/ope

#### <REF:DATA_LOAD>
> #### **3. Source Data Load**
>
> Load `data/data_clinpred.csv`, confirm the fields required by the Phase 1 builder exist, and keep the full dataframe available for calibration-gate diagnostics. This is offline analytical use only; production runtime uses the compact benchmark artifact.


In [3]:
# <REF:DATA_LOAD_CODE>
if not DATA_CLINPRED_PATH.exists():
    raise FileNotFoundError(f"Missing source data: {DATA_CLINPRED_PATH}")

df_full = pd.read_csv(DATA_CLINPRED_PATH, low_memory=False)
print(f"Loaded data_clinpred.csv: {df_full.shape[0]:,} rows x {df_full.shape[1]:,} columns")

PHASE1_REQUIRED_COLUMNS = [
    "nct_id",
    "enrollment",
    "enrollment_type",
    "overall_status",
    "phase",
    "phase_ml",
    "therapeutic_area",
    "therapeutic_area_ml",
    "gbd_cause_id_3_ml",
    "gbd_indication_name_3",
    "is_rare_disease",
    "is_rare_disease_ml",
]

schema_check = pd.DataFrame({
    "column": PHASE1_REQUIRED_COLUMNS,
    "present": [column in df_full.columns for column in PHASE1_REQUIRED_COLUMNS],
    "dtype": [str(df_full[column].dtype) if column in df_full.columns else "MISSING" for column in PHASE1_REQUIRED_COLUMNS],
    "missing_n": [int(df_full[column].isna().sum()) if column in df_full.columns else None for column in PHASE1_REQUIRED_COLUMNS],
})
display(schema_check)

missing_required = schema_check.loc[~schema_check["present"], "column"].tolist()
if missing_required:
    raise AssertionError(f"Missing required Phase 1 columns: {missing_required}")
# <REF:/DATA_LOAD_CODE>


Loaded data_clinpred.csv: 34,066 rows x 157 columns


,column,present,dtype,missing_n
0,nct_id,True,object,0
1,enrollment,True,float64,14
2,enrollment_type,True,object,14
3,overall_status,True,object,0
4,phase,True,object,0
5,phase_ml,True,int64,0
6,therapeutic_area,True,object,0
7,therapeutic_area_ml,True,int64,0
8,gbd_cause_id_3_ml,True,int64,0
9,gbd_indication_name_3,True,object,79


#### <REF:ENROLLMENT_AUDIT>
> #### **4. Enrollment Field Audit**
>
> Audit the source fields used by the enrollment benchmark: enrollment value/type, current trial status, phase, therapeutic area, GBD L3 indication, and rare-disease flag.
>
> The key separation is:
> - Completed positive `ACTUAL` enrollment: historical benchmark target.
> - `ESTIMATED` enrollment: planned/design-stage starting assumption.
> - Ongoing `ACTUAL` enrollment: observed lower bound, not final truth.


In [4]:
# <REF:ENROLLMENT_AUDIT_CODE>
AUDIT_COLUMNS = [
    "enrollment",
    "enrollment_type",
    "overall_status",
    "phase",
    "phase_ml",
    "therapeutic_area",
    "therapeutic_area_ml",
    "gbd_cause_id_3_ml",
    "gbd_indication_name_3",
    "is_rare_disease",
    "is_rare_disease_ml",
]

audit_rows = []
for column in AUDIT_COLUMNS:
    s = df_full[column]
    audit_rows.append({
        "column": column,
        "dtype": str(s.dtype),
        "missing_n": int(s.isna().sum()),
        "missing_pct": round(float(s.isna().mean()), 4),
        "n_unique": int(s.nunique(dropna=True)),
        "sample_values": ", ".join(map(str, s.dropna().astype(str).unique()[:6])),
    })

display(pd.DataFrame(audit_rows).sort_values(["missing_pct", "column"], ascending=[False, True]))

print("Enrollment type distribution:")
display(df_full["enrollment_type"].value_counts(dropna=False).rename_axis("enrollment_type").reset_index(name="n"))

print("Overall status distribution:")
display(df_full["overall_status"].value_counts(dropna=False).rename_axis("overall_status").reset_index(name="n"))

print("Phase distribution:")
display(df_full["phase"].value_counts(dropna=False).rename_axis("phase").reset_index(name="n"))

print("Therapeutic area distribution, top 20:")
display(df_full["therapeutic_area"].value_counts(dropna=False).head(20).rename_axis("therapeutic_area").reset_index(name="n"))

print("Enrollment distribution, positive values only:")
positive_enrollment = pd.to_numeric(df_full["enrollment"], errors="coerce")
positive_enrollment = positive_enrollment[positive_enrollment.gt(0)]
display(positive_enrollment.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("enrollment"))
# <REF:/ENROLLMENT_AUDIT_CODE>


,column,dtype,missing_n,missing_pct,n_unique,sample_values
8,gbd_indication_name_3,object,79,0.00,154,"Diarrheal diseases, Other malignant neoplasms,..."
0,enrollment,float64,14,0.00,2108,"1500.0, 73.0, 1072.0, 558.0, 55.0, 58.0"
1,enrollment_type,object,14,0.00,2,"ACTUAL, ESTIMATED"
7,gbd_cause_id_3_ml,int64,0,0.00,155,"302, 489, 408, 679, 981, 520"
9,is_rare_disease,int64,0,0.00,2,"0, 1"
10,is_rare_disease_ml,int64,0,0.00,2,"0, 1"
2,overall_status,object,0,0.00,7,"COMPLETED, TERMINATED, RECRUITING, WITHDRAWN, ..."
3,phase,object,0,0.00,4,"PHASE3, PHASE2, PHASE1/PHASE2, PHASE2/PHASE3"
4,phase_ml,int64,0,0.00,4,"4, 2, 1, 3"
5,therapeutic_area,object,0,0.00,19,"INFECTIONS, ONCOLOGY, OPHTHALMOLOGY, RESPIRATO..."


Enrollment type distribution:


,enrollment_type,n
0,ACTUAL,27506
1,ESTIMATED,6546
2,NaN,14


Overall status distribution:


,overall_status,n
0,COMPLETED,20719
1,TERMINATED,4195
2,RECRUITING,4134
3,ACTIVE_NOT_RECRUITING,2487
4,WITHDRAWN,1245
5,NOT_YET_RECRUITING,1139
6,ENROLLING_BY_INVITATION,147


...

,phase,n
0,PHASE2,14387
1,PHASE3,13884
2,PHASE1/PHASE2,4739
3,PHASE2/PHASE3,1056


Therapeutic area distribution, top 20:


,therapeutic_area,n
0,ONCOLOGY,8396
1,INFECTIONS,3158
2,NEUROLOGY,2599
3,DERMATOLOGY,2414
4,GASTROINTESTINAL,2290
5,METABOLIC,2110
6,RESPIRATORY,2001
7,MUSCULOSKELETAL,1972
8,CARDIOVASCULAR,1670
9,OPHTHALMOLOGY,1574


Enrollment distribution, positive values only:


,enrollment
count,"32,820.00"
mean,369.28
std,"1,477.05"
min,1.00
1%,3.00
5%,11.00
25%,50.00
50%,139.00
75%,344.00
90%,684.00


#### <REF:ENROLLMENT_FLAGS>
> #### **5. Enrollment Source and Readiness Flags**
>
> Reproduce the Phase 1 source/readiness flags using the same implementation functions as the build script.
>
> Completed actual enrollment can calibrate historical benchmark distributions. Ongoing actual enrollment is not final truth. Estimated/planned enrollment can seed the later Simulation Mode assumption. A user-edited value will later be labelled `user_scenario`.


In [5]:
# <REF:ENROLLMENT_FLAGS_CODE>
from scripts.build_enrollment_benchmarks import add_source_flags, load_source

benchmark_source = add_source_flags(load_source(DATA_CLINPRED_PATH))
flag_columns = [
    "is_completed_actual_enrollment_target",
    "is_estimated_planned_enrollment",
    "is_ongoing_actual_enrollment_lower_bound",
]

flag_summary = pd.DataFrame({
    "flag": flag_columns,
    "true_count": [int(benchmark_source[column].sum()) for column in flag_columns],
    "pct_of_records": [round(float(benchmark_source[column].mean()), 4) for column in flag_columns],
})
display(flag_summary)

print("Completed ACTUAL enrollment targets by phase:")
display(
    benchmark_source.loc[benchmark_source["is_completed_actual_enrollment_target"]]
    .groupby("phase")
    .size()
    .rename("benchmark_target_n")
    .reset_index()
    .sort_values("benchmark_target_n", ascending=False)
)

print("Estimated/planned enrollment availability by phase:")
display(
    benchmark_source.loc[benchmark_source["is_estimated_planned_enrollment"]]
    .groupby("phase")
    .size()
    .rename("estimated_planned_n")
    .reset_index()
    .sort_values("estimated_planned_n", ascending=False)
)
# <REF:/ENROLLMENT_FLAGS_CODE>


,flag,true_count,pct_of_records
0,is_completed_actual_enrollment_target,20526,0.60
1,is_estimated_planned_enrollment,6546,0.19
2,is_ongoing_actual_enrollment_lower_bound,1606,0.05


Completed ACTUAL enrollment targets by phase:


,phase,benchmark_target_n
3,PHASE3,9121
1,PHASE2,8987
0,PHASE1/PHASE2,1907
2,PHASE2/PHASE3,511


Estimated/planned enrollment availability by phase:


,phase,estimated_planned_n
1,PHASE2,2299
3,PHASE3,2277
0,PHASE1/PHASE2,1664
2,PHASE2/PHASE3,306


#### <REF:CALIBRATION_GATE>
> #### **6. Enrollment Benchmark Calibration Gate**
>
> This gate is practical and audit-oriented. It checks coverage, group sample sizes, rough effect size on `log1p(actual_enrollment)`, percentile spread, outlier sensitivity, fallback behavior, and label stability.
>
> It does not focus on p-values. The objective is stable benchmark construction, not hypothesis testing.
>
> Default v1 decision supported by the architecture:
>
> ```text
> Primary benchmark fields: phase + indication / therapeutic area + rare disease flag
> Other fields: support/conflict signals unless later proven stable enough for primary cohort matching
> ```


In [6]:
# <REF:CALIBRATION_GATE_CODE>
completed_actual = benchmark_source.loc[benchmark_source["is_completed_actual_enrollment_target"]].copy()
completed_actual["log1p_actual_enrollment"] = np.log1p(completed_actual["enrollment"])

candidate_fields = [
    "phase",
    "gbd_cause_id_3_ml",
    "therapeutic_area",
    "is_rare_disease_ml",
    "therapeutic_modality_ml",
    "sponsor_tier_ml",
    "administration_complexity_ml",
    "line_of_therapy_ml",
    "patient_severity_ml",
    "adult_ml",
    "child_ml",
    "older_adult_ml",
    "healthy_volunteers_ml",
    "endpoint_rigor_ml",
    "endpoint_structure_ml",
    "primary_duration_months_ml",
    "number_of_arms_ml",
    "comparator_benchmark_ml",
    "has_placebo_ml",
    "allocation_ml",
    "masking_ml",
]

# Attach optional candidate fields from the full source without changing the Phase 1 target logic.
optional_fields = [field for field in candidate_fields if field in df_full.columns and field not in completed_actual.columns]
completed_actual = completed_actual.join(df_full.loc[completed_actual.index, optional_fields])

calibration_rows = []
for field in candidate_fields:
    if field not in completed_actual.columns:
        calibration_rows.append({"field": field, "available": False})
        continue

    s = completed_actual[field]
    group_stats = (
        completed_actual.loc[s.notna()]
        .groupby(field, dropna=False)["enrollment"]
        .agg(benchmark_n="size", p25=lambda x: x.quantile(0.25), p50="median", p75=lambda x: x.quantile(0.75))
        .reset_index()
    )
    stable_groups = group_stats[group_stats["benchmark_n"].ge(50)]
    median_log = completed_actual.loc[s.notna()].groupby(field)["log1p_actual_enrollment"].median()
    effect_range = float(median_log.max() - median_log.min()) if len(median_log) else np.nan
    median_iqr = float((stable_groups["p75"] - stable_groups["p25"]).median()) if not stable_groups.empty else np.nan

    calibration_rows.append({
        "field": field,
        "available": True,
        "coverage_pct": round(float(s.notna().mean()), 4),
        "n_groups": int(s.nunique(dropna=True)),
        "median_group_n": float(group_stats["benchmark_n"].median()) if not group_stats.empty else np.nan,
        "groups_n_ge_50": int(stable_groups.shape[0]),
        "pct_groups_n_ge_50": round(float(stable_groups.shape[0] / len(group_stats)), 4) if len(group_stats) else np.nan,
        "log_median_effect_range": round(effect_range, 3) if pd.notna(effect_range) else np.nan,
        "median_iqr_enrollment_stable_groups": round(median_iqr, 2) if pd.notna(median_iqr) else np.nan,
    })

calibration_gate = pd.DataFrame(calibration_rows)
display(calibration_gate.sort_values(["available", "groups_n_ge_50", "coverage_pct"], ascending=[False, False, False]))

outlier_threshold = completed_actual["enrollment"].quantile(0.99)
outlier_summary = {
    "completed_actual_n": int(len(completed_actual)),
    "p99_enrollment": round(float(outlier_threshold), 2),
    "above_p99_count": int(completed_actual["enrollment"].gt(outlier_threshold).sum()),
    "max_enrollment": float(completed_actual["enrollment"].max()),
    "median_enrollment": float(completed_actual["enrollment"].median()),
}
print("Outlier sensitivity summary:")
print(json.dumps(outlier_summary, indent=2))

print("Calibration decision: keep the primary v1 hierarchy simple; retain non-primary fields for later support/conflict signals.")
# <REF:/CALIBRATION_GATE_CODE>


,field,available,coverage_pct,n_groups,median_group_n,groups_n_ge_50,pct_groups_n_ge_50,log_median_effect_range,median_iqr_enrollment_stable_groups
1,gbd_cause_id_3_ml,True,1.00,152,52.00,77,0.51,4.64,288.50
15,primary_duration_months_ml,True,1.00,851,1.00,59,0.07,9.02,290.75
2,therapeutic_area,True,1.00,19,"1,059.00",19,1.00,2.18,254.00
4,therapeutic_modality_ml,True,1.00,10,"1,080.50",10,1.00,2.55,220.25
16,number_of_arms_ml,True,1.00,30,12.50,10,0.33,3.97,329.00
7,line_of_therapy_ml,True,1.00,6,"1,623.00",6,1.00,1.80,279.50
8,patient_severity_ml,True,1.00,5,"2,478.00",5,1.00,0.88,265.00
20,masking_ml,True,1.00,5,"4,008.00",5,1.00,1.05,346.00
0,phase,True,1.00,4,"5,447.00",4,1.00,1.90,256.75
17,comparator_benchmark_ml,True,1.00,4,"5,646.50",4,1.00,1.57,304.12


Outlier sensitivity summary:
{
  "completed_actual_n": 20526,
  "p99_enrollment": 3548.75,
  "above_p99_count": 206,
  "max_enrollment": 90116.0,
  "median_enrollment": 154.0
}
Calibration decision: keep the primary v1 hierarchy simple; retain non-primary fields for later support/conflict signals.


#### <REF:BENCHMARK_COHORTS>
> #### **7. Benchmark Cohorts and Fallback Hierarchy**
>
> Build benchmark cohorts with the approved v1 hierarchy and the Phase 1 default threshold `n >= 50`.
>
> ```text
> Level 1: same phase + same indication + rare disease flag
> Level 2: same phase + same therapeutic area + rare disease flag
> Level 3: same phase + same therapeutic area
> Level 4: same phase only
> ```
>
> Runtime uses the strictest confident row available, then falls back to broader levels.


In [7]:
# <REF:BENCHMARK_COHORTS_CODE>
from scripts.build_enrollment_benchmarks import build_benchmarks

MIN_N = 50
in_memory_artifact = build_benchmarks(
    benchmark_source,
    min_n=MIN_N,
    source_data_version="notebook_in_memory",
    created_at="notebook_in_memory",
)

cohort_summary = (
    in_memory_artifact.groupby("benchmark_level_used")
    .agg(
        benchmark_rows=("benchmark_key", "size"),
        confident_rows=("low_confidence_flag", lambda x: int((~x).sum())),
        low_confidence_rows=("low_confidence_flag", "sum"),
        median_n=("benchmark_n", "median"),
        min_n=("benchmark_n", "min"),
        max_n=("benchmark_n", "max"),
    )
    .reset_index()
)
display(cohort_summary)

print(f"Sparse cohort rows (n < {MIN_N}): {int(in_memory_artifact['low_confidence_flag'].sum()):,}")
print("Fallback policy: choose first level with a non-low-confidence row; if none exists, return the broadest available low-confidence row or not_available.")
# <REF:/BENCHMARK_COHORTS_CODE>


,benchmark_level_used,benchmark_rows,confident_rows,low_confidence_rows,median_n,min_n,max_n
0,phase_indication_rare,658,104,554,7.00,1,647
1,phase_only,4,4,0,"5,447.00",511,9121
2,phase_ta,76,48,28,96.50,1,1570
3,phase_ta_rare,138,53,85,38.50,1,1422


Sparse cohort rows (n < 50): 667
Fallback policy: choose first level with a non-low-confidence row; if none exists, return the broadest available low-confidence row or not_available.


#### <REF:BENCHMARK_PERCENTILES>
> #### **8. Benchmark Percentiles**
>
> For each cohort, calculate `benchmark_n`, `benchmark_p25`, `benchmark_p50`, `benchmark_p75`, `benchmark_p90`, `low_confidence_flag`, and `benchmark_level_used`.
>
> Percentiles are used instead of means so very large trials do not dominate the benchmark.


In [8]:
# <REF:BENCHMARK_PERCENTILES_CODE>
percentile_columns = [
    "benchmark_level_used",
    "benchmark_key",
    "benchmark_n",
    "benchmark_p25",
    "benchmark_p50",
    "benchmark_p75",
    "benchmark_p90",
    "low_confidence_flag",
]

print("Generated benchmark table sample:")
display(in_memory_artifact[percentile_columns].head(12))

print("Percentile sanity summary across generated rows:")
display(
    in_memory_artifact[["benchmark_p25", "benchmark_p50", "benchmark_p75", "benchmark_p90"]]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
)
# <REF:/BENCHMARK_PERCENTILES_CODE>


Generated benchmark table sample:


,benchmark_level_used,benchmark_key,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,low_confidence_flag
0,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,138.00,240.00,342.00,403.20,True
1,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,71.00,88.00,105.00,115.20,True
2,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,26,26.25,36.00,52.00,178.00,True
3,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,3,116.00,132.00,356.00,490.40,True
4,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,7,69.00,102.00,358.50,569.60,True
5,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,37,48.00,183.00,600.00,682.80,True
6,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,32.50,36.00,39.50,41.60,True
7,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,43,105.00,276.00,534.00,847.60,True
8,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,4,166.00,696.50,"1,271.25","1,372.50",True
9,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,373.00,703.00,"1,033.00","1,231.00",True


Percentile sanity summary across generated rows:


,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90
count,876.00,876.00,876.00,876.00
mean,126.59,194.51,313.48,575.40
std,185.14,230.59,395.11,"1,643.57"
min,1.00,1.00,1.00,1.00
5%,13.00,20.00,29.62,36.75
25%,31.19,51.50,79.75,120.75
50%,63.50,111.50,189.00,256.15
75%,156.81,264.88,416.38,584.72
95%,398.69,627.00,985.75,"1,512.35"
max,"2,530.00","2,530.00","3,888.50","36,826.40"


#### <REF:BENCHMARK_ARTIFACT>
> #### **9. Compact Benchmark Artifact**
>
> Load the Phase 1 artifact from `frontend/data/enrollment_benchmarks_v1.csv` and the practical calibration report from `frontend/data/enrollment_benchmarks_v1_report.json`.
>
> Rebuild command:
>
> ```bash
> python scripts/build_enrollment_benchmarks.py
> ```


In [9]:
# <REF:BENCHMARK_ARTIFACT_CODE>
if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Missing benchmark artifact: {ARTIFACT_PATH}")
if not REPORT_PATH.exists():
    raise FileNotFoundError(f"Missing calibration report: {REPORT_PATH}")

artifact = pd.read_csv(ARTIFACT_PATH)
report = json.loads(REPORT_PATH.read_text())

print(f"Artifact path: {ARTIFACT_PATH.relative_to(project_root)}")
print(f"Report path:   {REPORT_PATH.relative_to(project_root)}")
print(f"Artifact rows: {len(artifact):,}")
print("Rebuild command: python scripts/build_enrollment_benchmarks.py")

print("Artifact schema:")
print("\n".join(artifact.columns))

print("Rows by benchmark level:")
display(artifact["benchmark_level_used"].value_counts().sort_index().rename_axis("benchmark_level_used").reset_index(name="rows"))

metadata_columns = ["benchmark_version", "source_data_version", "created_at", "outlier_policy", "calibration_notes"]
display(artifact[metadata_columns].drop_duplicates().head(10))

print("Report headline metrics:")
headline = {key: report[key] for key in ["records_loaded", "completed_actual_enrollment_targets", "sparse_cohort_count", "low_confidence_benchmark_count"]}
print(json.dumps(headline, indent=2))
# <REF:/BENCHMARK_ARTIFACT_CODE>


Artifact path: frontend/data/enrollment_benchmarks_v1.csv
Report path:   frontend/data/enrollment_benchmarks_v1_report.json
Artifact rows: 876
Rebuild command: python scripts/build_enrollment_benchmarks.py
Artifact schema:
benchmark_version
source_data_version
benchmark_key
phase
indication_or_therapeutic_area
gbd_cause_id_3_ml
therapeutic_area
rare_disease_flag
benchmark_level_used
benchmark_n
benchmark_p25
benchmark_p50
benchmark_p75
benchmark_p90
low_confidence_flag
created_at
outlier_policy
calibration_notes
Rows by benchmark level:


,benchmark_level_used,rows
0,phase_indication_rare,658
1,phase_only,4
2,phase_ta,76
3,phase_ta_rare,138


,benchmark_version,source_data_version,created_at,outlier_policy,calibration_notes
0,enrollment_benchmarks_v1,0a97519bd78f561a,2026-06-01T13:00:00+00:00,positive completed ACTUAL enrollment; percenti...,Deterministic v1 planned-enrollment benchmark;...


Report headline metrics:
{
  "records_loaded": 34066,
  "completed_actual_enrollment_targets": 20526,
  "sparse_cohort_count": 667,
  "low_confidence_benchmark_count": 667
}


#### <REF:ENROLLMENT_CLASSIFICATION>
> #### **10. Enrollment Status Classification**
>
> Demonstrate deterministic classification around P25, P75, and P90 boundaries. Boundaries are inclusive for `typical` at P25/P75 and inclusive for `ambitious` at P90.


In [10]:
# <REF:ENROLLMENT_CLASSIFICATION_CODE>
from src.enrollment_benchmarks import classify_enrollment

boundary_row = pd.Series({"benchmark_p25": 25, "benchmark_p50": 50, "benchmark_p75": 75, "benchmark_p90": 90})
examples = pd.DataFrame({"planned_enrollment": [24, 25, 50, 75, 76, 90, 91]})
examples["enrollment_status"] = examples["planned_enrollment"].map(lambda value: classify_enrollment(value, boundary_row))
display(examples)

print("Classification rule:")
print("< P25 -> below_benchmark | P25..P75 -> typical | >P75..P90 -> ambitious | >P90 -> above_benchmark_high")
# <REF:/ENROLLMENT_CLASSIFICATION_CODE>


,planned_enrollment,enrollment_status
0,24,below_benchmark
1,25,typical
2,50,typical
3,75,typical
4,76,ambitious
5,90,ambitious
6,91,above_benchmark_high


Classification rule:
< P25 -> below_benchmark | P25..P75 -> typical | >P75..P90 -> ambitious | >P90 -> above_benchmark_high


#### <REF:RUNTIME_LOOKUP_DEMO>
> #### **11. Runtime Lookup Demo**
>
> Demonstrate how production runtime uses only the current trial snapshot and compact artifact. It does not need the full historical `data_clinpred.csv` to calculate benchmarks.
>
> This demo uses the Phase 1 runtime utility from `src/enrollment_benchmarks.py`.


In [11]:
# <REF:RUNTIME_LOOKUP_DEMO_CODE>
from src.enrollment_benchmarks import (
    load_enrollment_benchmarks,
    lookup_enrollment_benchmark,
    planned_enrollment_metadata,
)

runtime_artifact = load_enrollment_benchmarks(ARTIFACT_PATH)
print(f"Runtime artifact loaded: {runtime_artifact.shape[0]:,} rows x {runtime_artifact.shape[1]:,} columns")

strict_row = runtime_artifact[
    runtime_artifact["benchmark_level_used"].eq("phase_indication_rare")
    & runtime_artifact["low_confidence_flag"].eq(False)
].iloc[0]
strict_snapshot = {
    "phase": strict_row["phase"],
    "gbd_cause_id_3_ml": int(strict_row["gbd_cause_id_3_ml"]),
    "therapeutic_area": strict_row.get("therapeutic_area"),
    "is_rare_disease_ml": int(strict_row["rare_disease_flag"]),
}
strict_lookup = lookup_enrollment_benchmark(strict_snapshot, runtime_artifact)
print("Strict lookup level:", strict_lookup["benchmark_level_used"])

fallback_seed = runtime_artifact[
    runtime_artifact["benchmark_level_used"].eq("phase_ta")
    & runtime_artifact["low_confidence_flag"].eq(False)
].iloc[0]
fallback_snapshot = {
    "phase": fallback_seed["phase"],
    "gbd_cause_id_3_ml": 999999999,
    "therapeutic_area": fallback_seed["therapeutic_area"],
    "is_rare_disease_ml": 1,
}
fallback_lookup = lookup_enrollment_benchmark(fallback_snapshot, runtime_artifact)
print("Fallback lookup level:", fallback_lookup["benchmark_level_used"] if fallback_lookup is not None else None)

with tempfile.TemporaryDirectory() as tmpdir:
    missing_artifact_metadata = planned_enrollment_metadata(
        strict_snapshot,
        600,
        artifact_path=Path(tmpdir) / "missing_enrollment_benchmarks.csv",
    )
print("Missing artifact status:", missing_artifact_metadata["planned_enrollment"]["enrollment_status"])

invalid_enrollment_metadata = planned_enrollment_metadata(strict_snapshot, None, artifact=runtime_artifact)
print("Missing enrollment status:", invalid_enrollment_metadata["planned_enrollment"]["enrollment_status"])

metadata_example = planned_enrollment_metadata(strict_snapshot, strict_lookup["benchmark_p50"], artifact=runtime_artifact)
display(pd.DataFrame([metadata_example["planned_enrollment"]]))
# <REF:/RUNTIME_LOOKUP_DEMO_CODE>


Runtime artifact loaded: 876 rows x 18 columns
Strict lookup level: phase_indication_rare
Fallback lookup level: phase_ta
Missing artifact status: not_available
Missing enrollment status: not_available


,value,source,benchmark_level_used,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,enrollment_status,support_level,supporting_signals,conflicting_signals,benchmark_snapshot_id,is_benchmark_stale,low_confidence_flag,interpretation_hint
0,100.00,planned_value,phase_indication_rare,65,40.00,100.00,140.00,224.60,typical,not_evaluated,[],[],enrollment_benchmarks_v1:0a97519bd78f561a:phas...,False,False,Enrollment is within the usual historical benc...


#### <REF:SNAPSHOT_METADATA>
> #### **12. Planned Snapshot Metadata Object**
>
> This object is what a later phase will attach to the latest prediction snapshot and pass into the narrative / Coherence layer.
>
> Phase 1 keeps `support_level` as `not_evaluated` and leaves support/conflict signal lists empty.


In [12]:
# <REF:SNAPSHOT_METADATA_CODE>
metadata = planned_enrollment_metadata(
    strict_snapshot,
    600,
    source="planned_value",
    artifact=runtime_artifact,
    is_benchmark_stale=False,
)
print(json.dumps(metadata, indent=2))
# <REF:/SNAPSHOT_METADATA_CODE>


{
  "planned_enrollment": {
    "value": 600.0,
    "source": "planned_value",
    "benchmark_level_used": "phase_indication_rare",
    "benchmark_n": 65,
    "benchmark_p25": 40.0,
    "benchmark_p50": 100.0,
    "benchmark_p75": 140.0,
    "benchmark_p90": 224.6,
    "enrollment_status": "above_benchmark_high",
    "support_level": "not_evaluated",
    "supporting_signals": [],
    "conflicting_signals": [],
    "benchmark_snapshot_id": "enrollment_benchmarks_v1:0a97519bd78f561a:phase_indication_rare|phase=PHASE1/PHASE2|indication=426|rare=0",
    "is_benchmark_stale": false,
    "low_confidence_flag": false,
    "interpretation_hint": "Enrollment is above the high historical benchmark for the matched cohort."
  }
}


#### <REF:VALIDATION>
> #### **13. Validation and Audit Summary**
>
> Summarize practical checks: sparse cohorts, low-confidence counts, fallback behavior, outlier sensitivity, label distribution, artifact schema, and consistency with the Phase 1 report JSON.
>
> Lightweight Phase 1 checks already run:
>
> ```bash
> python scripts/build_enrollment_benchmarks.py
> python scripts/check_enrollment_benchmarks.py
> python -m py_compile scripts/build_enrollment_benchmarks.py scripts/check_enrollment_benchmarks.py src/enrollment_benchmarks.py
> python audit_parity.py
> git diff --check
> ```


In [13]:
# <REF:VALIDATION_CODE>
required_artifact_columns = {
    "benchmark_version", "source_data_version", "benchmark_key", "phase", "indication_or_therapeutic_area",
    "gbd_cause_id_3_ml", "therapeutic_area", "rare_disease_flag", "benchmark_level_used", "benchmark_n",
    "benchmark_p25", "benchmark_p50", "benchmark_p75", "benchmark_p90", "low_confidence_flag", "created_at",
    "outlier_policy", "calibration_notes",
}
missing_artifact_columns = sorted(required_artifact_columns.difference(artifact.columns))
print("Missing artifact columns:", missing_artifact_columns)
assert not missing_artifact_columns

print("Report JSON consistency:")
print(f"Artifact rows: {len(artifact):,}")
print(f"Report low-confidence rows: {report['low_confidence_benchmark_count']:,}")
print(f"Artifact low-confidence rows: {int(artifact['low_confidence_flag'].astype(str).str.lower().isin(['true', '1', 'yes']).sum()):,}")

# Fast vectorized fallback audit for all completed ACTUAL targets.
confident_by_key = {
    row["benchmark_key"]: row
    for _, row in artifact.loc[~artifact["low_confidence_flag"].astype(str).str.lower().isin(["true", "1", "yes"])].iterrows()
}
all_by_key = {row["benchmark_key"]: row for _, row in artifact.iterrows()}

def key_for(level, row):
    if level == "phase_indication_rare":
        return f"{level}|phase={row['phase']}|indication={int(row['gbd_cause_id_3_ml'])}|rare={int(row['is_rare_disease_ml'])}"
    if level == "phase_ta_rare":
        return f"{level}|phase={row['phase']}|ta={row['therapeutic_area']}|rare={int(row['is_rare_disease_ml'])}"
    if level == "phase_ta":
        return f"{level}|phase={row['phase']}|ta={row['therapeutic_area']}"
    return f"{level}|phase={row['phase']}"

def fallback_row_for(row):
    levels = ["phase_indication_rare", "phase_ta_rare", "phase_ta", "phase_only"]
    keys = [key_for(level, row) for level in levels]
    for key in keys:
        if key in confident_by_key:
            return confident_by_key[key]
    for key in keys:
        if key in all_by_key:
            return all_by_key[key]
    return None

validation_targets = completed_actual.copy()
selected_rows = [fallback_row_for(row) for _, row in validation_targets.iterrows()]
validation_targets["fallback_level_used"] = [row["benchmark_level_used"] if row is not None else "not_available" for row in selected_rows]
validation_targets["enrollment_status"] = [
    classify_enrollment(validation_targets.iloc[i]["enrollment"], row) if row is not None else "not_available"
    for i, row in enumerate(selected_rows)
]

print("Fallback frequency across completed ACTUAL benchmark targets:")
display(validation_targets["fallback_level_used"].value_counts().rename_axis("benchmark_level_used").reset_index(name="n"))

print("Enrollment status distribution across completed ACTUAL benchmark targets:")
display(validation_targets["enrollment_status"].value_counts().rename_axis("enrollment_status").reset_index(name="n"))


print("Percentile spread stability proxy by level:")
artifact_spread = artifact.copy()
artifact_spread["relative_iqr_to_median"] = (artifact_spread["benchmark_p75"] - artifact_spread["benchmark_p25"]) / artifact_spread["benchmark_p50"].replace(0, np.nan)
display(
    artifact_spread.groupby("benchmark_level_used")["relative_iqr_to_median"]
    .agg(["count", "median", "mean", "max"])
    .reset_index()
)

print("Label stability proxy under +/-5% percentile-threshold stress:")
changed_under_stress = []
for i, row in enumerate(selected_rows):
    if row is None:
        changed_under_stress.append(True)
        continue
    enrollment_value = validation_targets.iloc[i]["enrollment"]
    base_status = classify_enrollment(enrollment_value, row)
    lower_thresholds = row.copy()
    upper_thresholds = row.copy()
    for col in ["benchmark_p25", "benchmark_p75", "benchmark_p90"]:
        lower_thresholds[col] = float(row[col]) * 0.95
        upper_thresholds[col] = float(row[col]) * 1.05
    changed_under_stress.append(
        classify_enrollment(enrollment_value, lower_thresholds) != base_status
        or classify_enrollment(enrollment_value, upper_thresholds) != base_status
    )

label_stability = pd.Series(changed_under_stress, name="label_changed_under_stress")
print({
    "evaluated_records": int(len(label_stability)),
    "changed_under_stress_n": int(label_stability.sum()),
    "changed_under_stress_pct": round(float(label_stability.mean()), 4),
})

print("Sparse and low-confidence summary:")
display(
    artifact.groupby("benchmark_level_used")
    .agg(
        rows=("benchmark_key", "size"),
        low_confidence_rows=("low_confidence_flag", lambda x: int(x.astype(str).str.lower().isin(["true", "1", "yes"]).sum())),
        median_n=("benchmark_n", "median"),
    )
    .reset_index()
)

print("Outlier summary from report JSON:")
print(json.dumps(report["outlier_summary"], indent=2))

print("Limitations: percentiles are historical benchmarks, not clinical recommendations; support/conflict logic and Coherence scoring are Phase 2+ work.")
# <REF:/VALIDATION_CODE>


Missing artifact columns: []
Report JSON consistency:
Artifact rows: 876
Report low-confidence rows: 667
Artifact low-confidence rows: 667
Fallback frequency across completed ACTUAL benchmark targets:


,benchmark_level_used,n
0,phase_indication_rare,15038
1,phase_ta_rare,4117
2,phase_ta,809
3,phase_only,562


Enrollment status distribution across completed ACTUAL benchmark targets:


,enrollment_status,n
0,typical,10274
1,below_benchmark,5324
2,ambitious,2960
3,above_benchmark_high,1968


Percentile spread stability proxy by level:


,benchmark_level_used,count,median,mean,max
0,phase_indication_rare,658,0.88,0.92,5.50
1,phase_only,4,1.63,1.60,1.75
2,phase_ta,76,1.31,1.34,2.72
3,phase_ta_rare,138,1.19,1.20,4.06


Label stability proxy under +/-5% percentile-threshold stress:
{'evaluated_records': 20526, 'changed_under_stress_n': 2151, 'changed_under_stress_pct': 0.1048}
Sparse and low-confidence summary:


,benchmark_level_used,rows,low_confidence_rows,median_n
0,phase_indication_rare,658,554,7.00
1,phase_only,4,0,"5,447.00"
2,phase_ta,76,28,96.50
3,phase_ta_rare,138,85,38.50


Outlier summary from report JSON:
{
  "max_completed_actual_enrollment": 90116.0,
  "very_large_completed_actual_count": 206,
  "very_large_threshold_p99": 3548.75
}
Limitations: percentiles are historical benchmarks, not clinical recommendations; support/conflict logic and Coherence scoring are Phase 2+ work.


#### <REF:SITES_SOURCE_CONTRACT>
> #### **14. Site-Count Source Contract and Quality Audit**
>
> Bring the S1B site-count audit into the main estimation notebook. The important rule is terminology: `number_of_facilities` is a registry-derived aggregate facility-count proxy. It is not true planned sites, true actual activated sites, or true estimated sites.
>
> This section recalculates the core quality numbers so the site-count benchmark can be inspected next to enrollment.

In [14]:
# <REF:SITES_SOURCE_CONTRACT_CODE>
SITE_REQUIRED_COLUMNS = [
    "nct_id",
    "number_of_facilities",
    "overall_status",
    "phase",
    "phase_ml",
    "therapeutic_area",
    "therapeutic_area_ml",
    "gbd_cause_id_3_ml",
    "gbd_indication_name_3",
    "is_rare_disease_ml",
    "is_rare_disease",
]
missing_site_columns = [column for column in SITE_REQUIRED_COLUMNS if column not in df_full.columns]
if missing_site_columns:
    raise ValueError(f"Missing site-count source columns: {missing_site_columns}")

site_audit = df_full[SITE_REQUIRED_COLUMNS].copy()
site_audit["number_of_facilities"] = pd.to_numeric(site_audit["number_of_facilities"], errors="coerce")

def numeric_summary(series: pd.Series) -> dict:
    clean = pd.to_numeric(series, errors="coerce")
    return {
        "total_rows": int(len(clean)),
        "present": int(clean.notna().sum()),
        "missing": int(clean.isna().sum()),
        "zero": int(clean.eq(0).sum()),
        "negative": int(clean.lt(0).sum()),
        "positive": int(clean.gt(0).sum()),
        "median": float(clean.quantile(0.5)) if clean.notna().any() else None,
        "p90": float(clean.quantile(0.9)) if clean.notna().any() else None,
        "p95": float(clean.quantile(0.95)) if clean.notna().any() else None,
        "p99": float(clean.quantile(0.99)) if clean.notna().any() else None,
        "max": float(clean.max()) if clean.notna().any() else None,
    }

site_quality_stats = numeric_summary(site_audit["number_of_facilities"])
print(json.dumps(site_quality_stats, indent=2))

site_audit.groupby("overall_status", dropna=False)["number_of_facilities"].agg(
    rows="size",
    positive=lambda s: int(pd.to_numeric(s, errors="coerce").gt(0).sum()),
    median="median",
    p90=lambda s: float(pd.to_numeric(s, errors="coerce").quantile(0.9)),
).sort_values("rows", ascending=False).head(12)
# <REF:/SITES_SOURCE_CONTRACT_CODE>

{
  "total_rows": 34066,
  "present": 34066,
  "missing": 0,
  "zero": 2242,
  "negative": 0,
  "positive": 31824,
  "median": 12.0,
  "p90": 93.0,
  "p95": 149.0,
  "p99": 302.34999999999854,
  "max": 1745.0
}


,rows,positive,median,p90
overall_status,,,,
COMPLETED,20719,19880,13.00,90.00
TERMINATED,4195,4079,14.00,92.00
RECRUITING,4134,4134,10.00,104.00
ACTIVE_NOT_RECRUITING,2487,2485,28.00,175.00
WITHDRAWN,1245,560,0.00,9.60
NOT_YET_RECRUITING,1139,539,0.00,5.20
ENROLLING_BY_INVITATION,147,147,10.00,92.80


#### <REF:SITES_FLAGS_AND_TARGET>
> #### **15. Site-Count Source Flags and Benchmark Target**
>
> Reproduce the S2 source/readiness flags using the production builder logic.
>
> Site-count benchmark target:
>
> ```text
> overall_status == COMPLETED
> number_of_facilities > 0
> ```
>
> Interpretation: completed registry facility-count proxy values, not true actual activated site counts.

In [15]:
# <REF:SITES_FLAGS_AND_TARGET_CODE>
from scripts.build_site_benchmarks import add_source_flags as add_site_source_flags
from scripts.build_site_benchmarks import build_benchmarks as build_site_benchmarks
from scripts.build_site_benchmarks import load_source as load_site_source

site_benchmark_source = add_site_source_flags(load_site_source(DATA_CLINPRED_PATH))
completed_positive_sites = site_benchmark_source.loc[
    site_benchmark_source["is_completed_positive_site_count_target"]
].copy()

site_flag_summary = pd.DataFrame([
    {
        "flag": "is_completed_positive_site_count_target",
        "rows": int(site_benchmark_source["is_completed_positive_site_count_target"].sum()),
    },
    {
        "flag": "is_current_registry_facility_count_proxy",
        "rows": int(site_benchmark_source["is_current_registry_facility_count_proxy"].sum()),
    },
])
print(site_flag_summary.to_string(index=False))

site_target_summary = {
    "completed_positive_site_count_targets": int(len(completed_positive_sites)),
    "median": float(completed_positive_sites["number_of_facilities"].median()),
    "p90": float(completed_positive_sites["number_of_facilities"].quantile(0.9)),
    "max": float(completed_positive_sites["number_of_facilities"].max()),
}
print(json.dumps(site_target_summary, indent=2))
# <REF:/SITES_FLAGS_AND_TARGET_CODE>

                                    flag  rows
 is_completed_positive_site_count_target 19880
is_current_registry_facility_count_proxy  7305
{
  "completed_positive_site_count_targets": 19880,
  "median": 15.0,
  "p90": 92.0,
  "max": 1611.0
}


#### <REF:SITES_BENCHMARK_COHORTS>
> #### **16. Site-Count Benchmark Cohorts and Percentiles**
>
> Build the in-memory site benchmark with the same four-level fallback hierarchy and `min_n = 50` threshold used by enrollment. This mirrors the production `site_benchmarks_v1.csv` artifact.

In [16]:
# <REF:SITES_BENCHMARK_COHORTS_CODE>
site_in_memory_artifact = build_site_benchmarks(
    site_benchmark_source,
    min_n=MIN_N,
    source_data_version="notebook_in_memory",
)

site_level_summary = site_in_memory_artifact.groupby("benchmark_level_used").agg(
    rows=("benchmark_key", "count"),
    confident=("low_confidence_flag", lambda s: int((~s).sum())),
    low_confidence=("low_confidence_flag", lambda s: int(s.sum())),
    p50_min=("benchmark_p50", "min"),
    p50_median=("benchmark_p50", "median"),
    p50_max=("benchmark_p50", "max"),
).reset_index()
site_level_summary
# <REF:/SITES_BENCHMARK_COHORTS_CODE>

,benchmark_level_used,rows,confident,low_confidence,p50_min,p50_median,p50_max
0,phase_indication_rare,659,105,554,1.00,12.00,277.00
1,phase_only,4,4,0,5.00,12.50,28.00
2,phase_ta,76,48,28,1.00,11.75,75.00
3,phase_ta_rare,138,53,85,1.00,12.00,78.00


#### <REF:SITES_RUNTIME_ARTIFACT>
> #### **17. Site-Count Runtime Artifact and Metadata**
>
> Load the compact production site artifact and demonstrate runtime lookup/classification. The metadata is operational-assumption context only; it stays outside the prediction model and does not change Completion Score.

In [17]:
# <REF:SITES_RUNTIME_ARTIFACT_CODE>
from src.site_benchmarks import (
    classify_site_count,
    load_site_benchmarks,
    lookup_site_benchmark,
    planned_sites_metadata,
)

if not SITE_ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Missing site benchmark artifact: {SITE_ARTIFACT_PATH}")
if not SITE_REPORT_PATH.exists():
    raise FileNotFoundError(f"Missing site benchmark report: {SITE_REPORT_PATH}")

site_runtime_artifact = load_site_benchmarks(SITE_ARTIFACT_PATH)
site_report = json.loads(SITE_REPORT_PATH.read_text())
print("Site artifact rows:", len(site_runtime_artifact))
print("Rows by level:", site_report.get("benchmark_rows_by_level"))
print("Coverage QA:", site_report.get("coverage_qa_match_counts"))

site_strict_row = site_runtime_artifact[
    site_runtime_artifact["benchmark_level_used"].eq("phase_indication_rare")
    & site_runtime_artifact["low_confidence_flag"].eq(False)
].iloc[0]
site_strict_snapshot = {
    "phase": site_strict_row["phase"],
    "gbd_cause_id_3_ml": int(site_strict_row["gbd_cause_id_3_ml"]),
    "therapeutic_area": site_strict_row.get("therapeutic_area"),
    "is_rare_disease_ml": int(site_strict_row["rare_disease_flag"]),
}
site_lookup = lookup_site_benchmark(site_strict_snapshot, site_runtime_artifact)
print("Lookup level:", site_lookup["benchmark_level_used"], "n=", int(site_lookup["benchmark_n"]))

site_boundary_row = pd.Series({"benchmark_p25": 25, "benchmark_p75": 75, "benchmark_p90": 90})
pd.DataFrame({"planned_sites": [None, 0, 24, 25, 75, 76, 90, 91]}).assign(
    site_count_status=lambda d: d["planned_sites"].apply(lambda value: classify_site_count(value, site_boundary_row))
)
# <REF:/SITES_RUNTIME_ARTIFACT_CODE>

Site artifact rows: 877
Rows by level: {'phase_indication_rare': 659, 'phase_only': 4, 'phase_ta': 76, 'phase_ta_rare': 138}
Coverage QA: {'phase_indication_rare': 23439, 'phase_only': 1136, 'phase_ta': 2231, 'phase_ta_rare': 7260}
Lookup level: phase_indication_rare n= 64


,planned_sites,site_count_status
0,NaN,not_available
1,0.00,not_available
2,24.00,below_benchmark
3,25.00,typical
4,75.00,typical
5,76.00,ambitious
6,90.00,ambitious
7,91.00,above_benchmark_high


#### <REF:OPERATIONAL_BENCHMARK_ARTIFACT>
> #### **18. Combined Operational Benchmark Artifact**
>
> The combined artifact is the point where enrollment and sites come together. It carries enrollment, site-count proxy, and patients-per-site percentiles on the same benchmark keys.
>
> This lets the app derive a site default that is coherent with the selected planned enrollment without making `planned_sites` model-facing.

In [18]:
# <REF:OPERATIONAL_BENCHMARK_ARTIFACT_CODE>
from scripts.build_operational_benchmarks import add_source_flags as add_operational_source_flags
from scripts.build_operational_benchmarks import build_benchmarks as build_operational_benchmarks
from scripts.build_operational_benchmarks import load_source as load_operational_source
from src.operational_benchmarks import (
    load_operational_benchmarks,
    lookup_operational_benchmark,
    planned_sites_default_from_operational_benchmark,
)

operational_source = add_operational_source_flags(load_operational_source(DATA_CLINPRED_PATH))
operational_in_memory_artifact = build_operational_benchmarks(
    operational_source,
    min_n=MIN_N,
    source_data_version="notebook_in_memory",
)

if not OPERATIONAL_ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Missing operational benchmark artifact: {OPERATIONAL_ARTIFACT_PATH}")
if not OPERATIONAL_REPORT_PATH.exists():
    raise FileNotFoundError(f"Missing operational benchmark report: {OPERATIONAL_REPORT_PATH}")

operational_artifact = load_operational_benchmarks(OPERATIONAL_ARTIFACT_PATH)
operational_report = json.loads(OPERATIONAL_REPORT_PATH.read_text())

print("Operational artifact rows:", len(operational_artifact))
print("Rows by level:", operational_report.get("benchmark_rows_by_level"))
print("Targets:", {
    "completed_actual_enrollment_targets": operational_report.get("completed_actual_enrollment_targets"),
    "completed_positive_site_count_targets": operational_report.get("completed_positive_site_count_targets"),
    "completed_patients_per_site_targets": operational_report.get("completed_patients_per_site_targets"),
})
print("Low-confidence rows by metric:", operational_report.get("low_confidence_rows_by_metric"))
print("Coverage QA:", operational_report.get("coverage_qa"))

operational_artifact[[
    "benchmark_level_used",
    "benchmark_key",
    "enrollment_n",
    "enrollment_p50",
    "site_count_n",
    "site_count_p50",
    "patients_per_site_n",
    "patients_per_site_p50",
]].head(10)
# <REF:/OPERATIONAL_BENCHMARK_ARTIFACT_CODE>

Operational artifact rows: 878
Rows by level: {'phase_indication_rare': 660, 'phase_only': 4, 'phase_ta': 76, 'phase_ta_rare': 138}
Targets: {'completed_actual_enrollment_targets': 20526, 'completed_positive_site_count_targets': 19880, 'completed_patients_per_site_targets': 19689}
Low-confidence rows by metric: {'enrollment': 667, 'patients_per_site': 666, 'site_count': 667}
Coverage QA: {'enrollment': {'low_confidence_matches': 0, 'match_counts': {'phase_indication_rare': 23299, 'phase_only': 1136, 'phase_ta': 2270, 'phase_ta_rare': 7361}, 'not_available': 0}, 'patients_per_site': {'low_confidence_matches': 0, 'match_counts': {'phase_indication_rare': 23299, 'phase_only': 1136, 'phase_ta': 2270, 'phase_ta_rare': 7361}, 'not_available': 0}, 'site_count': {'low_confidence_matches': 0, 'match_counts': {'phase_indication_rare': 23439, 'phase_only': 1136, 'phase_ta': 2231, 'phase_ta_rare': 7260}, 'not_available': 0}}


,benchmark_level_used,benchmark_key,enrollment_n,enrollment_p50,site_count_n,site_count_p50,patients_per_site_n,patients_per_site_p50
0,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,240.00,2,13.00,2,18.25
1,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,88.00,2,2.00,2,47.33
2,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,26,36.00,25,2.00,25,24.00
3,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,3,132.00,3,1.00,3,100.00
4,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,7,102.00,7,1.00,7,102.00
5,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,37,183.00,35,13.00,35,19.71
6,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,2,36.00,2,24.00,2,1.49
7,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,43,276.00,43,4.00,40,55.45
8,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,0,NaN,1,4.00,0,NaN
9,phase_indication_rare,phase_indication_rare|phase=PHASE1/PHASE2|indi...,4,696.50,4,24.00,4,78.58


#### <REF:PLANNED_SITES_DEFAULTING>
> #### **19. Revised Planned-Sites Defaulting Logic**
>
> This is the current end-to-end rule for keeping sites realistic relative to enrollment.
>
> For completed trials, the site value can use the completed registry facility-count proxy. For non-completed trials, current registry facility count is a lower-bound/context candidate, not the planned-sites estimate by itself.
>
> When patients-per-site P50 is available, the selected non-completed default is the maximum of:
>
> 1. current registry facility-count proxy,
> 2. planned enrollment divided by matched patients-per-site P50.
>
> Pure matched site-count P50 is used only as fallback when patients-per-site P50 is unavailable.
>
> The ratio uses median patients-per-site from completed trials, not `enrollment_p50 / site_count_p50`. Median of trial-level ratios is safer than ratio of medians.

In [19]:
# <REF:PLANNED_SITES_DEFAULTING_CODE>
def row_to_benchmark_snapshot(row: pd.Series | dict) -> dict:
    snapshot = dict(row)
    if "phase_ml" in snapshot:
        snapshot["phase"] = snapshot.get("phase_ml")
    if "therapeutic_area_ml" in snapshot:
        snapshot["therapeutic_area"] = snapshot.get("therapeutic_area_ml")
    return snapshot

nct_example = "NCT02615184"
example_row = df_full[df_full["nct_id"].eq(nct_example)].iloc[0]
example_snapshot = row_to_benchmark_snapshot(example_row)
example_default = planned_sites_default_from_operational_benchmark(
    example_snapshot,
    planned_enrollment=example_row.get("enrollment"),
    current_registry_facility_count_proxy=example_row.get("number_of_facilities"),
    overall_status=example_row.get("overall_status"),
    artifact=operational_artifact,
)
print(nct_example)
print(json.dumps(example_default, indent=2))

hypothetical_default = planned_sites_default_from_operational_benchmark(
    example_snapshot,
    planned_enrollment=200,
    current_registry_facility_count_proxy=2,
    overall_status="NOT_YET_RECRUITING",
    artifact=operational_artifact,
)
print()
print("Hypothetical planned-stage trial: 2 current registry facilities, 200 planned enrollment")
print(json.dumps(hypothetical_default, indent=2))

pd.DataFrame([
    {"scenario": nct_example, **example_default},
    {"scenario": "hypothetical_2_sites_200_enrollment", **hypothetical_default},
])
# <REF:/PLANNED_SITES_DEFAULTING_CODE>

NCT02615184
{
  "value": 25,
  "source": "current_registry_facility_count_proxy",
  "site_default_basis": "current_registry_facility_count_proxy",
  "current_registry_facility_count_proxy": 25.0,
  "site_count_benchmark_p50": 10.0,
  "patients_per_site_p50": 7.31,
  "enrollment_coherent_site_candidate": 10.39671682626539,
  "operational_benchmark_snapshot_id": "operational_benchmarks_v1:0a97519bd78f561a:phase_indication_rare|phase=PHASE2|indication=992|rare=0"
}

Hypothetical planned-stage trial: 2 current registry facilities, 200 planned enrollment
{
  "value": 28,
  "source": "enrollment_coherent_benchmark_default",
  "site_default_basis": "enrollment_coherent_benchmark_default",
  "current_registry_facility_count_proxy": 2.0,
  "site_count_benchmark_p50": 10.0,
  "patients_per_site_p50": 7.31,
  "enrollment_coherent_site_candidate": 27.359781121751027,
  "operational_benchmark_snapshot_id": "operational_benchmarks_v1:0a97519bd78f561a:phase_indication_rare|phase=PHASE2|indication=992

,scenario,value,source,site_default_basis,current_registry_facility_count_proxy,site_count_benchmark_p50,patients_per_site_p50,enrollment_coherent_site_candidate,operational_benchmark_snapshot_id
0,NCT02615184,25,current_registry_facility_count_proxy,current_registry_facility_count_proxy,25.00,10.00,7.31,10.40,operational_benchmarks_v1:0a97519bd78f561a:pha...
1,hypothetical_2_sites_200_enrollment,28,enrollment_coherent_benchmark_default,enrollment_coherent_benchmark_default,2.00,10.00,7.31,27.36,operational_benchmarks_v1:0a97519bd78f561a:pha...


#### <REF:ALL_ARTIFACT_VALIDATION>
> #### **20. All Benchmark Validation Commands**
>
> The notebook is explanatory. The source-of-truth validation remains the script layer:
>
> ```bash
> python scripts/check_enrollment_benchmarks.py
> python scripts/check_site_benchmarks.py
> python scripts/check_operational_benchmarks.py
> ```
>
> The code below performs notebook-local schema and consistency checks against the generated artifacts.

In [20]:
# <REF:ALL_ARTIFACT_VALIDATION_CODE>
assert len(runtime_artifact) > 0
assert len(site_runtime_artifact) > 0
assert len(operational_artifact) > 0
assert operational_report["duplicate_benchmark_keys"] == 0
assert operational_report["source_records_loaded"] == len(df_full)

for metric in ["enrollment", "site_count", "patients_per_site"]:
    assert operational_report["coverage_qa"][metric]["not_available"] == 0
    assert operational_report["coverage_qa"][metric]["low_confidence_matches"] == 0

required_operational_columns = {
    "benchmark_version", "source_data_version", "benchmark_key", "phase", "benchmark_level_used",
    "enrollment_n", "enrollment_p25", "enrollment_p50", "enrollment_p75", "enrollment_p90",
    "site_count_n", "site_count_p25", "site_count_p50", "site_count_p75", "site_count_p90",
    "patients_per_site_n", "patients_per_site_p25", "patients_per_site_p50", "patients_per_site_p75", "patients_per_site_p90",
}
missing_operational_columns = required_operational_columns.difference(operational_artifact.columns)
assert not missing_operational_columns, missing_operational_columns

validation_summary = {
    "enrollment_artifact_rows": int(len(runtime_artifact)),
    "site_artifact_rows": int(len(site_runtime_artifact)),
    "operational_artifact_rows": int(len(operational_artifact)),
    "operational_duplicate_keys": int(operational_report["duplicate_benchmark_keys"]),
    "operational_rows_by_level": operational_report["benchmark_rows_by_level"],
    "operational_low_confidence_rows_by_metric": operational_report["low_confidence_rows_by_metric"],
}
print(json.dumps(validation_summary, indent=2))
# <REF:/ALL_ARTIFACT_VALIDATION_CODE>

{
  "enrollment_artifact_rows": 876,
  "site_artifact_rows": 877,
  "operational_artifact_rows": 878,
  "operational_duplicate_keys": 0,
  "operational_rows_by_level": {
    "phase_indication_rare": 660,
    "phase_only": 4,
    "phase_ta": 76,
    "phase_ta_rare": 138
  },
  "operational_low_confidence_rows_by_metric": {
    "enrollment": 667,
    "patients_per_site": 666,
    "site_count": 667
  }
}


#### <REF:CONCLUSION_AND_NEXT_STEPS>
> #### **21. Conclusion and Next Steps**
>
> This notebook now shows the estimation logic in one place:
>
> - Planned Enrollment benchmark.
> - Planned Site Count benchmark.
> - Combined operational benchmark.
> - Revised non-completed planned-sites defaulting rule.
>
> Current production/runtime boundary remains unchanged: these operational assumptions do not enter XGBoost, SHAP, `/predict`, prediction payloads, calibration, audit parity, API contracts, model artifacts, or taxonomy artifacts.
>
> Next practical step is browser smoke testing the Simulation Mode site card/default behavior.

In [21]:
# <REF:CONCLUSION_AND_NEXT_STEPS_CODE>
print("Enrollment artifact:", ARTIFACT_PATH.relative_to(project_root))
print("Site artifact:", SITE_ARTIFACT_PATH.relative_to(project_root))
print("Operational artifact:", OPERATIONAL_ARTIFACT_PATH.relative_to(project_root))
print("Enrollment runtime utility:", RUNTIME_UTILITY_PATH.relative_to(project_root))
print("Site runtime utility:", SITE_RUNTIME_UTILITY_PATH.relative_to(project_root))
print("Operational runtime utility:", OPERATIONAL_RUNTIME_UTILITY_PATH.relative_to(project_root))
print("Primary rebuild commands:")
print("  python scripts/build_enrollment_benchmarks.py")
print("  python scripts/build_site_benchmarks.py")
print("  python scripts/build_operational_benchmarks.py")
print("Primary validation commands:")
print("  python scripts/check_enrollment_benchmarks.py")
print("  python scripts/check_site_benchmarks.py")
print("  python scripts/check_operational_benchmarks.py")
print("Notebook complete: enrollment, sites, and combined operational benchmark logic are now documented together.")
# <REF:/CONCLUSION_AND_NEXT_STEPS_CODE>


Enrollment artifact: frontend/data/enrollment_benchmarks_v1.csv
Site artifact: frontend/data/site_benchmarks_v1.csv
Operational artifact: frontend/data/operational_benchmarks_v1.csv
Enrollment runtime utility: src/enrollment_benchmarks.py
Site runtime utility: src/site_benchmarks.py
Operational runtime utility: src/operational_benchmarks.py
Primary rebuild commands:
  python scripts/build_enrollment_benchmarks.py
  python scripts/build_site_benchmarks.py
  python scripts/build_operational_benchmarks.py
Primary validation commands:
  python scripts/check_enrollment_benchmarks.py
  python scripts/check_site_benchmarks.py
  python scripts/check_operational_benchmarks.py
Notebook complete: enrollment, sites, and combined operational benchmark logic are now documented together.
